# 07 FS2 Prophet

`Prophet` joins only from `FS2` onward.

Why the delay is deliberate:
- `Prophet` is more meaningful once calendar and holiday structure is part of the design
- treating `Prophet` as an `FS1` lag-only model would not be methodologically fair
- `FS2` is therefore the first valid entry point for `Prophet` in this DAM stack


In [ ]:
from pathlib import Path
import os
import subprocess
import sys
import time

import pandas as pd
from IPython.display import Markdown, display

NOTEBOOK_CWD = Path.cwd()
REPO_ROOT = next(
    path for path in [NOTEBOOK_CWD, *NOTEBOOK_CWD.parents] if (path / "scripts/Data/02_Forecasting/01_DA_prices").exists()
)
os.chdir(REPO_ROOT)
PACKAGE_ROOT = REPO_ROOT / "scripts/Data/02_Forecasting/01_DA_prices"
if str(PACKAGE_ROOT) not in sys.path:
    sys.path.append(str(PACKAGE_ROOT))

from hourly_da.core.config import HourlyDAPipelineConfig
from hourly_da.core.external_features import build_external_family_catalog, load_external_feature_store
from hourly_da.core.methodology import (
    STARTER_ENDOGENOUS_FEATURE_NOTE,
    feature_stage_policy_frame,
    model_status_frame,
    shortlisting_policy_frame,
)
from hourly_da.core.reporting import find_latest_run, load_csv, load_json
from hourly_da.core.tuning import build_tuning_placeholder, tuning_cadence_frame, tuning_snippet_frame
from hourly_da.notebook_support import estimate_run_duration_seconds, format_duration, load_selected_case_weeks

config = HourlyDAPipelineConfig(
    input_csv=REPO_ROOT / "data/01_cleaned/Day_ahead_prices/DA_prices/hourly/da_prices_all_regions_hourly.csv",
    raw_root=REPO_ROOT / "data/00_Raw/DA_Prices",
    cleaned_feature_root=REPO_ROOT / "data/01_cleaned",
    output_root=REPO_ROOT / "data/02_Forecasting/01_DA_prices/hourly_da",
)
output_root = config.output_root


def latest_run_or_none(run_label: str) -> Path | None:
    try:
        return find_latest_run(output_root, run_label)
    except FileNotFoundError:
        return None


In [ ]:
display(
    feature_stage_policy_frame()
    .loc[lambda df: df["fs_level"] == "FS2"]
    .reset_index(drop=True)
)


In [ ]:
display(
    model_status_frame()
    .loc[lambda df: df["model_family"].isin(['prophet'])]
    .reset_index(drop=True)
)


In [ ]:
display(pd.DataFrame([build_tuning_placeholder("prophet", "FS2")]))
display(tuning_snippet_frame(model_family="prophet", fs_level="FS2"))


In [ ]:
run_dir = latest_run_or_none("prophet_benchmark")
focus_models = ['naive_previous_day', 'naive_previous_week', 'naive_previous_year', 'prophet_fs2']

if run_dir is None:
    print("No saved run exists yet for this notebook under the finalized methodology.")
else:
    print(run_dir)
    metrics_by_reporting_level = load_csv(run_dir, "metrics_by_reporting_level.csv")
    timing_summary = load_csv(run_dir, "origin_timing_summary.csv")
    official_naive = load_json(run_dir, "official_naive_reference.json")
    display(
        metrics_by_reporting_level[metrics_by_reporting_level["model"].isin(focus_models)]
        .sort_values(["dataset_split", "reporting_level_sort_order", "mae", "model"])
        .reset_index(drop=True)
    )
    display(pd.DataFrame([official_naive]))
    display(
        timing_summary[timing_summary["model"].isin(focus_models)]
        .sort_values(["dataset_split", "fit_time_mean_sec", "model"])
        .reset_index(drop=True)
    )


Later execution note:
- keep the `FS2` Prophet regressor set compact and causal
- use validation-only structural tuning
- do not backfill Prophet into `FS1`


In [ ]:
ALLOW_HEAVY_RERUN = False

if ALLOW_HEAVY_RERUN:
    estimate = estimate_run_duration_seconds(output_root, "prophet_benchmark")
    if estimate is not None:
        print(
            "Heavy rerun warning: latest comparable run "
            f"{estimate['run_id']} suggests about {format_duration(float(estimate['estimate_seconds']))}."
        )
    else:
        print("Heavy rerun warning: no comparable runtime estimate was found for this stage.")

    command = [sys.executable, str(PACKAGE_ROOT / "run_prophet_benchmark.py")]

    started = time.perf_counter()
    subprocess.run(command, check=True)
    elapsed_seconds = time.perf_counter() - started
    print(f"Actual wall-clock time: {format_duration(elapsed_seconds)}")
else:
    print("Rerun skipped. Set ALLOW_HEAVY_RERUN = True only when you are ready to execute the finalized pipeline.")
